# Tetris Fase 1A — Potencia en Colab

Este notebook clona el repo del proyecto, instala dependencias, corre las simulaciones para varios `tracker_prob`, y genera las curvas de potencia.

**Antes de empezar:** reemplaza `REPO_URL` en la celda siguiente por la URL de tu repo de GitHub.

## 1. Configuración

In [ ]:
# CONFIGURACIÓN
REPO_URL = "https://github.com/USUARIO/REPO.git"  # <-- REEMPLAZAR
PROJECT_DIR = "/content/tetris"
N_GAMES = 100              # reducir a 50 si el tiempo es crítico
MAX_PIECES = 500
TAU = 10.0
H_MIN = 4
H_MAX = 15
TRACKER_PROBS = [0.10, 0.25, 0.50, 0.75]

import os
print("Configuración:")
print(f"  REPO_URL={REPO_URL}")
print(f"  N_GAMES={N_GAMES}")
print(f"  MAX_PIECES={MAX_PIECES}")

## 2. Clonar repo e instalar dependencias

In [ ]:
import subprocess
import sys

# Clonar repo
if os.path.isdir(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}

# Instalar dependencias
!pip install -q pygame pandas pyarrow numpy scipy matplotlib tabulate statsmodels

## 3. Verificar estructura

In [ ]:
import os
conf_dir = os.path.join(PROJECT_DIR, "analysis", "confound_floor")
for f in ["confound_floor_t2.py", "power_curve_t2.py", "fast_calibrate_signal_v2.py", "recalc_power_calibrated.py"]:
    path = os.path.join(conf_dir, f)
    print("OK" if os.path.exists(path) else f"FALTA: {f}")

## 4. Correr simulaciones por tracker_prob

In [ ]:
import subprocess
import os
import time

conf_dir = os.path.join(PROJECT_DIR, "analysis", "confound_floor")
python = sys.executable

def run_simulation(tp, n_games=N_GAMES):
    tp_str = f"{int(tp*100):03d}"
    out_dir = f"out_t2_H15_n{n_games}_tp{tp_str}_nocensor"
    out_path = os.path.join(conf_dir, out_dir)
    cmd = [
        python, "confound_floor_t2.py",
        "--n_games", str(n_games),
        "--max_pieces", str(MAX_PIECES),
        "--tau", str(TAU),
        "--H_min", str(H_MIN),
        "--H_max", str(H_MAX),
        "--tracker_prob", str(tp),
        "--no_censorship",
        "--out", out_path
    ]
    print(f"\n[tracker_prob={tp}] Iniciando...")
    t0 = time.time()
    result = subprocess.run(cmd, cwd=conf_dir, capture_output=True, text=True)
    elapsed = time.time() - t0
    print(result.stdout)
    if result.returncode != 0:
        print(f"ERROR (returncode={result.returncode}):")
        print(result.stderr)
    else:
        print(f"[tracker_prob={tp}] OK en {elapsed/60:.1f} min")
    return result.returncode == 0

# Correr simulaciones
for tp in TRACKER_PROBS:
    run_simulation(tp)

## 5. Análisis de potencia (lineal)

In [ ]:
# Usar la corrida tracker_prob=0.5 como base
base_dir = os.path.join(conf_dir, f"out_t2_H15_n{N_GAMES}_tp050_nocensor")
power_out = os.path.join(conf_dir, "out_power_t2")

cmd = [
    python, "power_curve_t2.py",
    "--results_dir", base_dir,
    "--out", power_out
]
result = subprocess.run(cmd, cwd=conf_dir, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 6. Calibración rápida de la forma de β_señal(p)

In [ ]:
import pandas as pd
import numpy as np

# Crear muestra del 20% del log p=0.5 para acelerar la calibración
log_path = os.path.join(base_dir, "decisions_log_k5.parquet")
df = pd.read_parquet(log_path)

bins = [(4,6,'4-6'),(7,8,'7-8'),(9,10,'9-10'),(11,12,'11-12'),(13,15,'13-15')]
sampled_ids = []
for hmin, hmax, label in bins:
    ids = df[(df['H']>=hmin)&(df['H']<=hmax)]['decision_id'].unique()
    n_sample = max(100, int(len(ids)*0.2))
    sampled_ids.extend(np.random.choice(ids, size=min(n_sample, len(ids)), replace=False))

df_sample = df[df['decision_id'].isin(sampled_ids)].copy()
id_map = {old: new for new, old in enumerate(sorted(df_sample['decision_id'].unique()))}
df_sample['decision_id'] = df_sample['decision_id'].map(id_map)
sample_path = os.path.join(base_dir, "decisions_log_k5_sample20.parquet")
df_sample.to_parquet(sample_path)
print(f"Muestra guardada: {df_sample['decision_id'].nunique()} decisiones")

# Calibración rápida
cal_out = os.path.join(conf_dir, "out_fast_calibrate_v2")
bin_results = os.path.join(power_out, "bin_results.json")
cmd = [
    python, "fast_calibrate_signal_v2.py",
    "--log", sample_path,
    "--real_results", bin_results,
    "--out", cal_out,
    "--n_replicates", "5"
]
result = subprocess.run(cmd, cwd=conf_dir, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
# Recalcular p_min con curva calibrada
cal_json = os.path.join(cal_out, "fast_calibration_v2.json")
cmd = [
    python, "recalc_power_calibrated.py",
    "--bin_results", bin_results,
    "--calibration", cal_json
]
result = subprocess.run(cmd, cwd=conf_dir, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 7. Empaquetar y descargar resultados

In [ ]:
import shutil
import os

zip_path = os.path.join(PROJECT_DIR, "results_colab")
shutil.make_archive(
    zip_path, "zip",
    root_dir=conf_dir,
    base_dir="."
)
print(f"Zip creado: {zip_path}.zip")
print(f"Tamaño: {os.path.getsize(zip_path+'.zip')/1e6:.1f} MB")

In [ ]:
from google.colab import files
files.download(f"{zip_path}.zip")

## Nota de interpretación preliminar

Si los resultados replican lo observado en local, el `p_min` detectable con una campaña humana realista estará alrededor de 0.66–0.70 en el mejor bin (H=4–6) y `>1` en los demás. Eso apunta a que el cuello de botella es el número de decisiones informativas por sesión, no la extrapolación lineal de `β_señal(p)`.

Antes de saltar a un probe exógeno, la palanca sobre el denominador es densificar decisiones: concentrar en H=4–6, sesiones más largas, o un régimen que genere más estados con ≥2 alternativas viables y bolsa no degenerada.